In [4]:
import os
import chromadb
from pymongo import MongoClient
from google import genai
from google.genai import types
from dotenv import load_dotenv
from langchain_text_splitters import RecursiveCharacterTextSplitter
from tqdm import tqdm
import time

load_dotenv(override=True)
client = genai.Client()

mongo_client = MongoClient(os.getenv('MONGO_URI', 'mongodb://localhost:27017/'))
db = mongo_client.guardian_db

chroma_client = chromadb.PersistentClient(path="./chroma_db_advanced")
vector_collection = chroma_client.get_or_create_collection(name="target_advanced")

text_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", r"(?<=\. )", " ", ""],
    chunk_size=400,  # smaller chunk, more precision
    chunk_overlap=50,
    length_function=len,
    is_separator_regex=True
)

def get_embeddings_with_retry(texts):
    SUB_BATCH_SIZE = 100 
    all_embeddings = []

    for i in range(0, len(texts), SUB_BATCH_SIZE):
        batch = texts[i:i + SUB_BATCH_SIZE]
        while True:
            try:
                response = client.models.embed_content(
                    model='gemini-embedding-001',
                    contents=batch,
                    config=types.EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT")
                )
                all_embeddings.extend([emb.values for emb in response.embeddings])
                break
            except Exception as e:
                if "429" in str(e):
                    time.sleep(5) 
                else:
                    raise e
    return all_embeddings

def process_batch_advanced(articles):
    all_chunk_texts = []
    all_chunk_ids = []
    all_chunk_metas = []
    processed_article_ids = []

    for doc in articles:
        article_id = doc['id']
        headline = doc.get('fields', {}).get('headline', doc.get('webTitle', ''))
        date = doc.get('webPublicationDate', '')[:10]
        body = doc.get('fields', {}).get('bodyText', '')

        if not body or len(body.strip()) < 10:
            # NEU: Eigenes Flag 'embedded_advanced'
            db.articles.update_one({"id": article_id}, {"$set": {"embedded_advanced": True, "error": "Empty body"}})
            continue
            
        raw_chunks = text_splitter.split_text(body)
        for i, c in enumerate(raw_chunks):
            # Headline and Date as prefix
            all_chunk_texts.append(f"Title: {headline}\nDate: {date}\n\n{c}") 
            all_chunk_ids.append(f"{article_id}#chunk_{i}")
            # add metadata to later fatch whole article from MongoDB
            all_chunk_metas.append({"article_id": article_id}) 

        processed_article_ids.append(article_id)

    if not all_chunk_texts:
        return

    try:
        embeddings = get_embeddings_with_retry(all_chunk_texts)

        vector_collection.add(
            ids=all_chunk_ids,
            embeddings=embeddings,
            documents=all_chunk_texts,
            metadatas=all_chunk_metas
        )

        db.articles.update_many({"id": {"$in": processed_article_ids}}, {"$set": {"embedded_advanced": True}})
        
        time.sleep(0.1) 

    except Exception as e:
        print(f"Batch-Fehler: {e}")

def process_everything():
    total_remaining = db.articles.count_documents({"embedded_advanced": {"$ne": True}})
    print(f"Start Advanced Ingestion: {total_remaining} Artikel verbleibend.")
    pbar = tqdm(total=total_remaining, desc="Ingestion Advanced")

    while True:
        articles = list(db.articles.find({"embedded_advanced": {"$ne": True}}).limit(50))
        if not articles: break
        
        process_batch_advanced(articles)
        pbar.update(len(articles))
    
    pbar.close()

In [5]:
process_everything()

Start Advanced Ingestion: 21200 Artikel verbleibend.


Ingestion Advanced: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 21200/21200 [2:53:09<00:00,  2.04it/s]
